## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [2]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [3]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [4]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [256]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [42]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = list(sorted(set(sequence)))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {}
idx_to_char = {}

for idx, char in enumerate(vocab):
    char_to_idx[char] = idx
    idx_to_char[idx] = char

#TODO: Convert the entire text based data into numerical data
data = []

for char in sequence:
    data.append(char_to_idx[char])


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [7]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [ ]:
# This is Cell #7

#TODO: Set your model's hyperparameters
# THESE ARE FOR THE WARANDPEACE TEXT
sequence_length = 200  # Length of each input sequence
stride = 25            # Stride for creating sequences
embedding_dim = 128     # Dimension of character embeddings
hidden_size = 256      # Number of features in the hidden state of the RNN
learning_rate = 0.001  # Learning rate for the optimizer
num_epochs = 10         # Number of epochs to train
batch_size = 32        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> Here

- **`sequence_length` (200):**  
  This parameter determines the number of characters (or timesteps) in each input sequence presented to the RNN. Longer sequences can capture more context but may also increase the complexity of training.

- **`stride` (25):**  
  The stride defines the step size used when generating overlapping sequences from the input text data. A larger stride will result in fewer sequences, while a smaller stride increases the number of generated sequences, particularly for capturing detailed context.

- **`embedding_dim` (128):**  
  This parameter specifies the size of the embedding vectors for each character in the vocabulary. It determines how many features each character will be represented with in the input layer, allowing the model to capture relations between different characters.

- **`hidden_size` (256):**  
  The hidden size indicates the number of units in the hidden layer of the RNN. It directly affects the model's capacity to learn complex patterns; larger values can capture more intricate dependencies but may also increase the risk of overfitting.

- **`learning_rate` (0.001):**  
  The learning rate controls how much to change the model's weights in response to the gradient of the loss function during training. A suitable learning rate is essential to ensure convergence; too high may lead to overshooting the minimum, while too low may slow down training.

- **`num_epochs` (10):**  
  The number of epochs specifies how many times the entire training dataset will pass through the model during training. More epochs can lead to better model performance but also to overfitting if the model begins to memorize the training data rather than generalize.

- **`batch_size` (32):**  
  The batch size determines how many samples are processed before the model's internal parameters are updated. A larger batch size may speed up training but could also lead to less frequent updates, while smaller batches allow for more updates per epoch, which can help with convergence.

- **`vocab_size`:**  
  This is the total number of unique characters in the dataset, which informs the size of the output layer and the embedding layer. It is crucial for creating the character embeddings and for making predictions.

- **`input_size`:**  
  This parameter is usually equal to the size of the vocabulary (`vocab_size`) and represents the dimensionality of the input data to the model.

- **`output_size`:**  
  This is also equal to the size of the vocabulary and specifies the number of classes for output predictions. In a character-level model, it defines how many possible characters the model can predict.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [57]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [ ]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True
)

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [11]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [59]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size, embedding_dim=embedding_dim).to(device)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [34]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/3503 [00:00<?, ?it/s]/tmp/ipykernel_5743/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_5743/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 3503/3503 [08:17<00:00,  7.04it/s]


Epoch [1/10], Loss: 1.5688, Accuracy: 53.74%


Epoch 2/10: 100%|██████████| 3503/3503 [08:00<00:00,  7.28it/s]


Epoch [2/10], Loss: 1.3915, Accuracy: 58.70%


Epoch 3/10: 100%|██████████| 3503/3503 [07:59<00:00,  7.31it/s]


...

Epoch [8/10], Loss: 1.3300, Accuracy: 60.44%


Epoch 9/10: 100%|██████████| 3503/3503 [07:57<00:00,  7.33it/s]


Epoch [9/10], Loss: 1.3279, Accuracy: 60.49%


Epoch 10/10: 100%|██████████| 3503/3503 [07:57<00:00,  7.33it/s]

Epoch [10/10], Loss: 1.3261, Accuracy: 60.55%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [28]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

# the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` results

In [55]:
sequence = "abcdefghijklmnopqrstuvwxyz" * 100

In [56]:
# This is Cell #7

#TODO: Set your model's hyperparameters
# THESE ARE FOR THE abcde.. sequence
sequence_length = 100  # Length of each input sequence
stride = 10            # Stride for creating sequences
embedding_dim = 256     # Dimension of character embeddings
hidden_size = 512      # Number of features in the hidden state of the RNN
learning_rate = 0.001  # Learning rate for the optimizer
num_epochs = 10         # Number of epochs to train
batch_size = 16        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)

In [60]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/14 [00:00<?, ?it/s]/tmp/ipykernel_5743/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_5743/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 14/14 [00:01<00:00,  9.57it/s]


Epoch [1/10], Loss: 0.7429, Accuracy: 92.74%


Epoch 2/10: 100%|██████████| 14/14 [00:01<00:00,  9.40it/s]


Epoch [2/10], Loss: 0.0098, Accuracy: 99.38%


Epoch 3/10: 100%|██████████| 14/14 [00:01<00:00, 10.43it/s]


...

Epoch [8/10], Loss: 0.0003, Accuracy: 100.00%


Epoch 9/10: 100%|██████████| 14/14 [00:01<00:00,  8.99it/s]


Epoch [9/10], Loss: 0.0002, Accuracy: 100.00%


Epoch 10/10: 100%|██████████| 14/14 [00:01<00:00, 10.51it/s]

Epoch [10/10], Loss: 0.0002, Accuracy: 100.00%


### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [ ]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    hidden = model.init_hidden(batch_size)
    for batch_inputs, batch_targets in test_loader:
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
        output, hidden = model(batch_inputs, hidden)
        hidden = hidden.detach()
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))
        total_loss += loss.item()
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total elements in this batch
    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

/tmp/ipykernel_5743/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_5743/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.4371, Test Accuracy: 57.67%


# loss for sequence

In [61]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0

    hidden = model.init_hidden(batch_size)

    for batch_inputs, batch_targets in test_loader:
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        # Forward pass
        output, hidden = model(batch_inputs, hidden)

        # Detach hidden state to prevent gradient accumulation
        hidden = hidden.detach()

        # Compute loss
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))
        total_loss += loss.item()

        # Compute accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total elements in this batch

    # Calculate average loss and accuracy
    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Test Loss: 0.0001, Test Accuracy: 100.00%


/tmp/ipykernel_5743/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_5743/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [40]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    input_indices = [char_to_idx[char] for char in start_text.lower()]

    hidden = model.init_hidden(1)

    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)

    generated_sequence = start_text

    for _ in range(k):
        output, hidden = model(input_tensor, hidden)

        logits = output[:, -1, :]  # Shape [1, vocab_size]

        next_char_idx = sample_from_output(logits, temperature).item()

        next_char = idx_to_char[next_char_idx]

        generated_sequence += next_char

        input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
    
    return generated_sequence

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: mansion of the other an
Generated text: the people, devotedly, and
Generated text: weather to the treased 
Generated text: war is the countess was si
Generated text: command interncertain, 
Generated text: horrificulexedbetwee
Generated text: museum and down th
Generated text: horrificance of him 
Generated text: significance of the splendid
Generated text: plough the same pr


ValueError: invalid literal for int() with base 10: ''

# the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` text generation

In [62]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    input_indices = [char_to_idx[char] for char in start_text.lower()]

    hidden = model.init_hidden(1)

    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)

    generated_sequence = start_text

    for _ in range(k):
        output, hidden = model(input_tensor, hidden)

        logits = output[:, -1, :]  # Shape [1, vocab_size]

        next_char_idx = sample_from_output(logits, temperature).item()

        next_char = idx_to_char[next_char_idx]

        generated_sequence += next_char

        input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
    
    return generated_sequence

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: worldefghijklmnop
Generated text: warstuvwx
Generated text: ponderghi
Generated text: abcdefghi
Generated text: abcdefghi
Generated text: abcdefghi
Generated text: weaklmnopq


ValueError: invalid literal for int() with base 10: ''

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


### Report: My Experiments with RNN-based Text Generation

#### **Overview**
In this project, I trained an RNN model to generate text using two datasets: 
1. A repetitive sequence (`"abcdefghijklmnopqrstuvwxyz" * 100`).
2. Text from *War and Peace*.

---

### **Dataset 1: Repetitive Sequence (`"abcdefghijklmnopqrstuvwxyz" * 100`)**

#### **Hyperparameters**
Here are the hyperparameters I used for this dataset:
- `sequence_length = 100`
- `stride = 10`
- `embedding_dim = 256`
- `hidden_size = 512`
- `learning_rate = 0.001`
- `num_epochs = 10`
- `batch_size = 16`

#### **Results**
- **Test Loss**: `0.0001`
- **Test Accuracy**: `100.00%`

The model performed exceptionally well on this dataset, as expected. Since the dataset was highly repetitive and deterministic, the model easily memorized the pattern and achieved near-perfect results.

#### **Generated Text**
Here are some examples of text generated by the model:
- With `temperature = 1.0`:
  - "abcdefghi"
  - "worldefghijklmnop"
  - "ponderghi"
- With `temperature = 0.5`:
  - "abcdefghi"
- With `temperature = 1.5`:
  - "warstuvwx"
  - "weaklmnopq"

#### **Observations**
1. **Impact of Temperature**:
   - I noticed that changing the `temperature` parameter had very little impact on the generated text for this dataset. Since the pattern was repetitive, even higher randomness (e.g., `temperature = 1.5`) didn’t introduce much variation.
   - The text always filled in the alphabet sequence relative to the last character in the initial input, regardless of the temperature.

2. **Challenges**:
   - There weren’t any significant challenges for this dataset. The training process was quick (around 30 seconds per epoch), and the model converged easily. The simplicity of the dataset meant there wasn’t much room for the model to struggle or fail.

---

### **Dataset 2: *War and Peace***

#### **Hyperparameters**
For the *War and Peace* dataset, I used the following hyperparameters:
- `sequence_length = 200`
- `stride = 25`
- `embedding_dim = 128`
- `hidden_size = 256`
- `learning_rate = 0.001`
- `num_epochs = 10`
- `batch_size = 32`

#### **Results**
- **Test Loss**: `1.4371`
- **Test Accuracy**: `57.67%`

This dataset was much more challenging for the model. While it captured the style and vocabulary of the text, it struggled to achieve high accuracy or maintain coherence in long-term dependencies. The results were still decent, considering the complexity of the dataset.

#### **Generated Text**
Here are some examples of text generated by the model:
- With `temperature = 1.0`:
  - "mansion of the other an"
  - "the people, devotedly, and"
  - "war is the countess was si"
- With `temperature = 1.5`:
  - "horrificulexedbetwee"
- With `temperature = 0.5`:
  - "horrificance of him"
  - "significance of the splendid"
  - "plough the same pr"

#### **Observations**
1. **Impact of Temperature**:
   - Higher temperatures (e.g., `temperature = 1.5`) made the generated text more random, but it often became nonsensical, such as "horrificulexedbetwee."
   - Lower temperatures (e.g., `temperature = 0.5`) produced much more coherent and readable text. For example, "horrificance of him" and "significance of the splendid" were significantly more understandable.
   - At the default temperature (`temperature = 1.0`), the text was somewhat comprehensible but occasionally awkward. For example, "mansion of the other an" starts off well but ends abruptly.

2. **Challenges**:
   - **Long Training Times**: Training on this dataset was incredibly time-consuming. Each epoch took about 7 minutes, so I had to wait over an hour to complete all 10 epochs. This made experimentation difficult.
   - **Complexity of the Dataset**: The large vocabulary and intricate sentence structures in *War and Peace* added significant challenges. The model struggled with maintaining coherence over longer sequences and required careful tuning of hyperparameters.
   - **Balancing Randomness and Readability**: It was tricky to find the right temperature value for generating text that was both creative and understandable.

---

### **Comparison Between Datasets**

| Aspect                    | Repetitive Sequence       | *War and Peace*      |
|---------------------------|---------------------------|----------------------|
| **Test Loss**             | `0.0001`                 | `1.4371`            |
| **Test Accuracy**         | `100.00%`                | `57.67%`            |
| **Generated Text**        | Predictable and simple    | Rich but varied     |
| **Impact of Temperature** | Minimal                  | Significant         |
| **Training Time**         | ~1.5 seconds/epoch        | ~7 minutes/epoch    |
| **Challenges**            | None significant         | Long training times, maintaining coherence |

---

### **Reflections**

1. **Impact of Dataset Complexity**:
   - The simple repetitive sequence dataset allowed the model to memorize patterns perfectly and produce deterministic outputs. However, this didn’t showcase the full capabilities of the RNN.
   - The *War and Peace* dataset, on the other hand, highlighted the model’s ability to learn complex language patterns, but it also exposed its limitations in handling long-term dependencies and diverse vocabulary.

2. **Insights About Temperature**:
   - The `temperature` parameter was key to controlling randomness in text generation. Lower values (e.g., `0.5`) produced more coherent and realistic text, while higher values (e.g., `1.5`) introduced diversity but often led to gibberish.

3. **Challenges Faced**:
   - **Training Time**: The long training times for the *War and Peace* dataset made iterative experimentation difficult.
   - **Balancing Hyperparameters**: It was challenging to tune hyperparameters like `sequence_length`, `stride`, and `batch_size` to balance training speed and model performance.
   - **Maintaining Coherence**: Generating longer, coherent text was particularly hard, as the model struggled with remembering context over extended sequences.

4. **Key Learnings About RNNs**:
   - RNNs are effective for sequence modeling and can capture complex patterns in text, but their ability to maintain coherence diminishes with longer sequences.
   - Hyperparameter tuning plays a critical role in model performance, especially for datasets with diverse structures like *War and Peace*.
   - The temperature parameter is essential for exploring the trade-off between creativity and readability in text generation.

---

### **Thought Process during implementation**

For the **`generate_text` function**, the intuition behind this was to mimic how humans might write text — by taking an initial prompt and predicting one character at a time to construct coherent sequences. I started by converting the input text into indices using the predefined `char_to_idx` mapping to ensure the input was compatible with the model's expectations.

I then initialized the hidden state for a batch size of one, as text generation inherently operates on a single sequence. Using a loop, I repeatedly passed the input tensor through the model, extracting the logits for the last time step, which represent the raw scores for each character in the vocabulary. The decision to sample the next character index from these logits (instead of always taking the most probable character) was based on the need to balance randomness and structure in the generated text. To achieve this, I used the `sample_from_output` function, which incorporates temperature scaling. The intuition behind using temperature scaling was to add variability to the predictions when needed (higher temperature) or make the text more deterministic and focused (lower temperature). Each predicted character was appended to the growing sequence, and the newly generated character was fed back into the model as the next input.

For the **testing loop**, the primary goal was to evaluate the trained model's performance on unseen data. I started by initializing the hidden state for the given batch size, ensuring the model could process input sequences in batches. The intuition behind detaching the hidden state after each batch was to prevent gradients from propagating unnecessarily across batches, which could lead to memory overhead or incorrect gradient computations. During each batch, I forwarded the inputs through the model to get the predictions, then calculated the loss by flattening both the model's output and the targets. This step ensured the predictions and targets had the appropriate shape for the `CrossEntropyLoss` function, which computes the difference between the predicted probability distribution and the actual target indices.

To calculate accuracy, I compared the predicted indices (obtained by selecting the character with the highest probability) with the actual target indices. By accumulating the number of correct predictions and dividing by the total number of predictions, I was able to compute the accuracy for the entire test dataset. The intuition here was to quantify how often the model's predictions aligned with the expected outputs, providing a tangible measure of performance. After processing all batches, I computed the average loss and accuracy, which gave a clear and interpretable picture of how well the model generalized to new data.